# Minimum Flips to Make `(a | b) == c`

## Problem
Given three integers `a`, `b`, and `c`, find the minimum number of bit flips required in `a` and `b` so that:

```python
(a | b) == c
```

A flip means changing a bit from `0 -> 1` or `1 -> 0`.

## Useful Bit Operations

```python
n & 1        # Get the rightmost (LSB) bit
n >>= 1      # Shift right by one bit
(n >> i) & 1 # Get the ith bit
x ^ y        # 1 where x and y differ
x & y        # 1 where both bits are 1
~x           # Bitwise NOT
n.bit_count() # Number of 1 bits
```

## Solution 1 — Bit-by-Bit Traversal

At every bit position, compare the corresponding bits of `a`, `b`, and `c`.

### Case 1: `c` bit is `0`
Both `a` and `b` must be `0` because `1 | anything = 1`.

Therefore:
```python
flips += bit1 + bit2
```

### Case 2: `c` bit is `1`
At least one of `a` or `b` must be `1`. If both are `0`, one flip is needed.

In [ ]:
def min_flips_traversal(a: int, b: int, c: int) -> int:
    ans = 0

    while a or b or c:
        bit1 = a & 1
        bit2 = b & 1
        bit3 = c & 1

        if bit3 == 0:
            ans += bit1 + bit2
        elif bit1 == 0 and bit2 == 0:
            ans += 1

        a >>= 1
        b >>= 1
        c >>= 1

    return ans

### Truth Table

| a | b | c | Required flips |
|---|---|---|---:|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 |
| 1 | 0 | 0 | 1 |
| 1 | 1 | 0 | 2 |
| 0 | 0 | 1 | 1 |
| 0 | 1 | 1 | 0 |
| 1 | 0 | 1 | 0 |
| 1 | 1 | 1 | 0 |

**Complexity:** `O(log(max(a,b,c)))` time and `O(1)` extra space.

## Solution 2 — XOR + Bit Counting

First calculate:

```python
(a | b) ^ c
```

XOR produces `1` at every position where `(a | b)` differs from `c`. Each such position requires **at least one flip**.

### Special case

```text
a = 1
b = 1
c = 0
```

`(a | b) ^ c` detects one mismatch, but both `a` and `b` must change from `1 -> 0`, so we actually need **2 flips**.

The positions requiring this extra flip are found by:

```python
a & b & ~c
```

In [ ]:
def min_flips_xor(a: int, b: int, c: int) -> int:
    mismatch = (a | b) ^ c
    extra = a & b & ~c

    return mismatch.bit_count() + extra.bit_count()

### One-line version

```python
return ((a | b) ^ c).bit_count() + (a & b & ~c).bit_count()
```

### Key intuition

```text
XOR             -> Where is the OR result wrong?
a & b & ~c      -> Where is one EXTRA flip required?
```

## Interview Takeaway

A strong progression in an interview is:

1. Derive the **bit-by-bit solution** first.
2. Explain the two cases based on whether the target `c` bit is `0` or `1`.
3. Analyze the complexity.
4. If asked for a more concise bitwise solution, recognize that XOR finds mismatches.
5. Handle the `a=1, b=1, c=0` case with `a & b & ~c`.

**Memory shortcut:**

> XOR finds the wrong positions; AND finds the positions that need an extra flip.

In [ ]:
# Quick test
a, b, c = 2, 6, 5

print(min_flips_traversal(a, b, c))
print(min_flips_xor(a, b, c))

assert min_flips_traversal(a, b, c) == min_flips_xor(a, b, c)